In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import umap
from IPython.display import HTML

np.random.seed(42)

X, y = make_blobs(
    n_samples=1600,
    n_features=12,
    centers=8,
    cluster_std=[2.5, 2.8, 2.2, 3.0, 2.4, 2.7, 2.3, 2.6],
    random_state=42
)

X[:, 2] = X[:, 0] * 0.4 + np.random.normal(0, 1.2, size=X.shape[0])
X[:, 5] = np.sin(X[:, 1]) + np.random.normal(0, 0.4, size=X.shape[0])
X[:, 8] = X[:, 3] * X[:, 4] / 20 + np.random.normal(0, 0.8, size=X.shape[0])

X = StandardScaler().fit_transform(X)

pca_proj = PCA(n_components=2, random_state=42).fit_transform(X)

umap_proj = umap.UMAP(
    n_components=2,
    n_neighbors=20,
    min_dist=0.15,
    metric="euclidean",
    random_state=42
).fit_transform(X)

tsne_proj = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
).fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(12, 4.8), facecolor="white")

plt.subplots_adjust(top=0.80, bottom=0.20, wspace=0.28)

titles = ["PCA", "UMAP", "t-SNE"]
projections = [pca_proj, umap_proj, tsne_proj]

lims = []

for proj in projections:
    x_min, x_max = proj[:, 0].min(), proj[:, 0].max()
    y_min, y_max = proj[:, 1].min(), proj[:, 1].max()
    pad_x = (x_max - x_min) * 0.08
    pad_y = (y_max - y_min) * 0.08
    lims.append((x_min - pad_x, x_max + pad_x, y_min - pad_y, y_max + pad_y))

total_frames = 75

def draw_frame(frame):
    for ax in axes:
        ax.clear()

    fig.suptitle(
        "Redução de dimensionalidade: a escolha do método muda a leitura",
        fontsize=16,
        fontweight="bold",
        y=0.95
    )

    frac1 = min(1, max(0, (frame + 1) / 25))
    frac2 = min(1, max(0, (frame - 10) / 25))
    frac3 = min(1, max(0, (frame - 20) / 25))
    fracs = [frac1, frac2, frac3]

    for i, ax in enumerate(axes):
        proj = projections[i]
        frac = fracs[i]
        n_pts = max(5, int(len(proj) * frac))

        ax.scatter(
            proj[:n_pts, 0],
            proj[:n_pts, 1],
            c=y[:n_pts],
            s=8,
            alpha=0.65
        )

        x0, x1, y0, y1 = lims[i]
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)

        ax.set_title(titles[i], fontsize=13, fontweight="bold", pad=8)
        ax.grid(alpha=0.15)

        if i == 0:
            ax.set_xlabel("Componente 1", fontsize=9)
            ax.set_ylabel("Componente 2", fontsize=9)
        elif i == 1:
            ax.set_xlabel("Dimensão 1", fontsize=9)
            ax.set_ylabel("Dimensão 2", fontsize=9)
        else:
            ax.set_xlabel("Dimensão 1", fontsize=9)
            ax.set_ylabel("Dimensão 2", fontsize=9)

    if frame >= 52:
        fig.text(
            0.5, 0.025,
            "PCA tende a preservar estrutura global; UMAP e t-SNE destacam melhor relações locais.",
            ha="center", va="center", fontsize=10, fontweight="bold"
        )

anim = FuncAnimation(fig, draw_frame, frames=total_frames, interval=220, repeat=True)

HTML(anim.to_jshtml())